This notebook filters Yellow Taxi and FHVHV to the zones obtained from the previous notebook and filters to the date range too (2023 and 2024)

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = SparkSession.builder \
    .appName("Taxi_Filtering") \
    .master("local[2]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()  #to handle large datasets

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 00:52:24 WARN Utils: Your hostname, LAPTOP-V00NHVKA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/31 00:52:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 00:52:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/31 00:52:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Reads the theater zones and finds all distinct values.

In [6]:
theaters_zones = pd.read_csv(
    "../data/broadway_data/theaters_zones.csv"
)

theater_zone_ids = (
    theaters_zones["LocationID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

theater_zone_ids

[230, 48, 100, 142]

TLC changes a column's data type sometimes(e.g. passenger_count switches between INT64 and DOUBLE), so we read one month as a reference schema first.

In [7]:
YEARS = ['2023', '2024']
MONTHS = range(1, 13)

yellow_schema = spark.read.parquet("../data/yellow_taxi_data/2023-01.parquet").schema
yellow_schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', DoubleType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', DoubleType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True)])

Read every month and cast every column to match the reference schema, and save each month into the raw layer.

In [4]:
import os

os.makedirs("../data/raw/yellow_taxi", exist_ok=True)

for YEAR in YEARS:
    for month in MONTHS:
        month = str(month).zfill(2)
        input_path = f"../data/yellow_taxi_data/{YEAR}-{month}.parquet"
        output_path = f"../data/raw/yellow_taxi/{YEAR}-{month}"

        sdf = spark.read.parquet(input_path)
        sdf = sdf.select([F.col(field.name).cast(field.dataType) for field in yellow_schema])

        sdf.write.mode("overwrite").parquet(output_path)

print("All months recast to a consistent schema and saved to ../data/raw/yellow_taxi")

All months recast to a consistent schema and saved to ../data/raw/yellow_taxi


Read the whole raw layer and print it to confirm.

In [8]:
yellow = spark.read.schema(yellow_schema).parquet("../data/raw/yellow_taxi/*")
yellow.printSchema()

26/08/31 00:52:43 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/yellow_taxi/*.
java.io.FileNotFoundException: File ../data/raw/yellow_taxi/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.anal

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



Count the Yellow Taxi data rows before filtering.

In [9]:
yellow_before = yellow.count()

print("Yellow Taxi rows before filtering:", yellow_before)

Yellow Taxi rows before filtering: 79479946


Trip distance and duration: a trip cannot have zero or negative distance, and a trip under 60 seconds is most likely a meter error than a real trip.

Check the percentiles for a reasonable bound.

In [7]:
yellow.select("trip_distance").summary("2%", "5%", "25%", "50%", "75%", "95%", "98%", "99.9%", "max").show()

+-------+-------------+
|summary|trip_distance|
+-------+-------------+
|     2%|         0.01|
|     5%|         0.44|
|    25%|         1.03|
|    50%|         1.77|
|    75%|          3.4|
|    95%|         14.9|
|    98%|        18.54|
|  99.9%|        29.96|
|    max|    398608.62|
+-------+-------------+



0.2 ~ 35 miles is decided as a reasonable distance bound and 1 minute ~ 2 hours is decided as the duration bound.

In [10]:
trip_duration_secs = (
    F.unix_timestamp("tpep_dropoff_datetime")
    - F.unix_timestamp("tpep_pickup_datetime")
)

yellow = yellow.filter(
    (F.col("trip_distance") > 0.2)
    &
    (F.col("trip_distance") < 35)
    &
    (trip_duration_secs >= 60)
    &
    (trip_duration_secs <= 7200)
)

Financial fields: no field representing a charge can be negative, and the metered fare cannot be below the initial unit charge ($3.00 for the whole 2023-2024 period).

In [ ]:
yellow = yellow.filter(
    (F.col("fare_amount") >= 3.00)
    &
    (F.col("extra") >= 0)
    &
    (F.col("mta_tax") >= 0)
    &
    (F.col("tip_amount") >= 0)
    &
    (F.col("tolls_amount") >= 0)
    &
    (F.col("total_amount") >= 0)
)
print("Yellow Taxi rows before filtering:", yellow_before)
print("Yellow Taxi rows after data-contraint filtering:", yellow.count())

Yellow Taxi rows before filtering: 79479946


Yellow Taxi rows after data-contraint filtering: 76110915


26/08/31 03:24:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3945146 ms exceeds timeout 120000 ms
26/08/31 03:24:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/08/31 05:06:58 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:70

Filter the Yellow Taxi data to only match the zones with theaters in them and the date range 2023-2024.

In [ ]:
yellow_filtered = yellow.filter(
    (
        F.col("PULocationID").isin(theater_zone_ids)
    )
    &
    (
        F.col("tpep_pickup_datetime") >= "2023-01-01"
    )
    &
    (
        F.col("tpep_pickup_datetime") < "2025-01-01"
    )
)

Prints the row count before and after filtering, also prints the percentage of rows left after filtering.

In [11]:
yellow_after = yellow_filtered.count()

print("Yellow Taxi rows before filtering:", yellow_before)
print("Yellow Taxi rows after filtering:", yellow_after)
print("Percentage retained:",round(yellow_after / yellow_before * 100, 2),"%")

Yellow Taxi rows before filtering: 79479946
Yellow Taxi rows after filtering: 8228548
Percentage retained: 10.35 %


FHVHV: do the exact same schema-fixing steps for this data too.

In [12]:
fhvhv_schema = spark.read.parquet("../data/fhvhv_data/2023-01.parquet").schema
fhvhv_schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_re

Recast every month to the reference schema and save to the raw layer.

In [13]:
os.makedirs("../data/raw/fhvhv", exist_ok=True)

for YEAR in YEARS:
    for month in MONTHS:
        month = str(month).zfill(2)
        input_path = f"../data/fhvhv_data/{YEAR}-{month}.parquet"
        output_path = f"../data/raw/fhvhv/{YEAR}-{month}"

        sdf = spark.read.parquet(input_path)
        sdf = sdf.select([F.col(field.name).cast(field.dataType) for field in fhvhv_schema])

        sdf.write.mode("overwrite").parquet(output_path)

print("All months recast to a consistent schema and saved to ../data/raw/fhvhv")

All months recast to a consistent schema and saved to ../data/raw/fhvhv


Read the whole raw layer.

In [14]:
fhvhv = spark.read.schema(fhvhv_schema).parquet("../data/raw/fhvhv/*")
fhvhv.printSchema()

26/08/30 21:00:30 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/fhvhv/*.
java.io.FileNotFoundException: File ../data/raw/fhvhv/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.Resolve

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

Count FHVHV rows before filtering.

In [15]:
fhvhv_before = fhvhv.count()

print("FHVHV rows before filtering:", fhvhv_before)

FHVHV rows before filtering: 471960468


FHVHV has different fields to Yellow Taxi, but we can still filter to the same constraints.

In [16]:
fhvhv_valid = fhvhv.filter(
    (F.col("trip_miles") > 0.2)
    &
    (F.col("trip_miles") < 35)
    &
    (F.col("trip_time") >= 60)
    &
    (F.col("trip_time") <= 7200)
    &
    (F.col("base_passenger_fare") >= 0)
    &
    (F.col("tolls") >= 0)
    &
    (F.col("bcf") >= 0)
    &
    (F.col("sales_tax") >= 0)
    &
    (F.col("tips") >= 0)
    &
    (F.col("driver_pay") >= 0)
)

fhvhv_quality_before = fhvhv.count()
fhvhv_quality_after = fhvhv_valid.count()
print(f"FHVHV rows before data-quality filtering: {fhvhv_quality_before}")
print(f"FHVHV rows after data-quality filtering:  {fhvhv_quality_after}")
print(f"Percentage retained: {round(fhvhv_quality_after / fhvhv_quality_before * 100, 2)}%")

fhvhv = fhvhv_valid

FHVHV rows before data-quality filtering: 471960468
FHVHV rows after data-quality filtering:  469327480
Percentage retained: 99.44%


Filter FHVHV to the same theater/control zones and date range.

In [17]:
fhvhv_filtered = fhvhv.filter(
    (
        F.col("PULocationID").isin(theater_zone_ids)
    )
    &
    (
        F.col("pickup_datetime") >= "2023-01-01"
    )
    &
    (
        F.col("pickup_datetime") < "2025-01-01"
    )
)

Count FHVHV rows after filtering.

In [18]:
fhvhv_after = fhvhv_filtered.count()

print("FHVHV rows before filtering:", fhvhv_before)
print("FHVHV rows after filtering:", fhvhv_after)
print("Percentage retained:",round(fhvhv_after / fhvhv_before * 100, 2),"%")


FHVHV rows before filtering: 471960468
FHVHV rows after filtering: 16632799
Percentage retained: 3.52 %


Control zone: pick a nearby Manhattan zone with no Broadway theaters to compare against later.

In [19]:
zone_lookup = pd.read_csv("../data/taxi_zones/taxi+_zone_lookup.csv")

control_zone_name = "Murray Hill"  # a zone not in theater_zone_ids

control_zone_id = int(
    zone_lookup.loc[zone_lookup["Zone"] == control_zone_name, "LocationID"].iloc[0]
)
assert control_zone_id not in theater_zone_ids, "Control zone overlaps theater zones"
print(f"Control zone: {control_zone_name} (LocationID {control_zone_id})")

Control zone: Murray Hill (LocationID 170)


Filter Yellow Taxi to just the control zone.

In [20]:
control_filtered = yellow.filter(
    (F.col("PULocationID") == control_zone_id)
    &
    (F.col("tpep_pickup_datetime") >= "2023-01-01")
    &
    (F.col("tpep_pickup_datetime") < "2025-01-01")
)

control_after = control_filtered.count()
print("Control zone rows before filtering:", yellow_before)
print("Control zone rows after filtering:", control_after)
print("Percentage retained:", round(control_after / yellow_before * 100, 2), "%")

Control zone rows before filtering: 79479946
Control zone rows after filtering: 2226344
Percentage retained: 2.8 %


Save the control zone data.

In [21]:
control_filtered.write.mode("overwrite").parquet("../data/curated/control_zone")

Filter FHVHV to the control zone too.

In [22]:
fhvhv_control_filtered = fhvhv.filter(
    (F.col("PULocationID") == control_zone_id)
    &
    (F.col("pickup_datetime") >= "2023-01-01")
    &
    (F.col("pickup_datetime") < "2025-01-01")
)

fhvhv_control_after = fhvhv_control_filtered.count()
print("FHVHV control zone rows before filtering:", fhvhv_before)
print("FHVHV control zone rows after filtering:", fhvhv_control_after)
print("Percentage retained:", round(fhvhv_control_after / fhvhv_before * 100, 2), "%")

FHVHV control zone rows before filtering: 471960468
FHVHV control zone rows after filtering: 4373056
Percentage retained: 0.93 %


Save the FHVHV control zone data.

In [23]:
fhvhv_control_filtered.write.mode("overwrite").parquet("../data/curated/fhvhv_control_zone")

Save the filtered data for future use.

In [24]:
yellow_filtered.write.mode("overwrite").parquet("../data/curated/yellow_taxi")
fhvhv_filtered.write.mode("overwrite").parquet("../data/curated/fhvhv")